### 2.1 理论计算题

给定一个字符序列"ababc"，假设采用一阶马尔可夫模型（即 $p(x_t|x_{t-1})$）使用拉普拉斯平滑（加1平滑）估计以下条件概率：

1. $p(a'|b')$ 
2. $p(c'|b')$

（词汇表为 $\{a', b', c'\}$，计算时考虑所有可能转移，包括未出现的情况。）

**解答：**

序列 "ababc" 中，观察所有相邻转移对：
- a → b (位置1: a→b)
- b → a (位置2: b→a)
- a → b (位置3: a→b)
- b → c (位置4: b→c)

统计转移计数：
- count(b → a) = 1
- count(b → c) = 1
- count(b → b) = 0

从状态 b 出发的总转移次数：$N_b = 1 + 1 + 0 = 2$

词汇表大小 $V = 3$（a, b, c）

使用拉普拉斯平滑（加1平滑）：
$$p(x_t|x_{t-1}) = \frac{\text{count}(x_{t-1} \rightarrow x_t) + 1}{\sum_{x' \in V} (\text{count}(x_{t-1} \rightarrow x') + 1)} = \frac{\text{count}(x_{t-1} \rightarrow x_t) + 1}{N_{x_{t-1}} + V}$$

1. $p(a'|b') = \frac{\text{count}(b \rightarrow a) + 1}{N_b + V} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$

2. $p(c'|b') = \frac{\text{count}(b \rightarrow c) + 1}{N_b + V} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$

In [2]:
import re
from collections import Counter, defaultdict

def preprocess_text(text, n):
    """
    预处理文本，构建词汇表，并生成n-gram特征序列和标签
    
    参数:
        text: 输入文本字符串
        n: 滑动窗口大小（特征序列长度）
    
    返回:
        vocab: 词汇表字典 {词: ID}
        features: 特征列表，每个特征为长度为n的词列表
        labels: 标签列表，每个标签为下一个词的ID
    """
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    words = [w for w in text.split() if w]
    
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    features = []
    labels = []
    
    for i in range(len(words) - n):
        feature = words[i:i+n]
        label_word = words[i+n]
        features.append(feature)
        labels.append(vocab[label_word])
    
    return vocab, features, labels


# 测试示例
if __name__ == "__main__":
    # 测试用例
    text = "The time machine"
    n = 2
    
    vocab, features, labels = preprocess_text(text, n)
    
    print("=" * 50)
    print("2.2 序列模型 - 编程题")
    print("=" * 50)
    print(f"输入文本: '{text}'")
    print(f"n = {n}")
    print(f"\n词汇表: {vocab}")
    print(f"\n特征列表: {features}")
    print(f"标签列表: {labels}")
    print(f"\n标签对应的词: {[list(vocab.keys())[list(vocab.values()).index(l)] for l in labels]}")
    
    # 额外测试
    print("\n" + "-" * 30)
    print("额外测试:")
    text2 = "hello world how are you today"
    n = 3
    vocab2, features2, labels2 = preprocess_text(text2, n)
    print(f"文本: '{text2}', n={n}")
    print(f"词汇表: {vocab2}")
    print(f"特征: {features2}")
    print(f"标签: {labels2}")

2.2 序列模型 - 编程题
输入文本: 'The time machine'
n = 2

词汇表: {'machine': 0, 'the': 1, 'time': 2}

特征列表: [['the', 'time']]
标签列表: [0]

标签对应的词: ['machine']

------------------------------
额外测试:
文本: 'hello world how are you today', n=3
词汇表: {'are': 0, 'hello': 1, 'how': 2, 'today': 3, 'world': 4, 'you': 5}
特征: [['hello', 'world', 'how'], ['world', 'how', 'are'], ['how', 'are', 'you']]
标签: [0, 5, 3]


### 3.1 理论计算题

考虑一个线性RNN（无偏置），定义为 $h_t = W_{hh}h_{t-1} + W_{hx}x_t$，输出 $o_t = W_{oh}h_t$。假设损失函数为平方损失 $L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2$。推导损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

**解答：**

设损失函数为：
$$L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2 = \frac{1}{2}\sum_{t=1}^{T}(W_{oh}h_t - y_t)^2$$

对于每个时间步 t，定义误差项：
$$\delta_t = \frac{\partial L}{\partial h_t}$$

根据链式法则，在时间步 t 的误差由两部分组成：
1. 当前时间步输出对损失的贡献
2. 未来时间步通过 $W_{hh}$ 传递回来的误差

$$\delta_t = \frac{\partial L}{\partial o_t}\frac{\partial o_t}{\partial h_t} + \frac{\partial L}{\partial h_{t+1}}\frac{\partial h_{t+1}}{\partial h_t}$$

其中：
$$\frac{\partial L}{\partial o_t} = o_t - y_t$$
$$\frac{\partial o_t}{\partial h_t} = W_{oh}^T$$
$$\frac{\partial h_{t+1}}{\partial h_t} = W_{hh}^T$$

因此：
$$\delta_t = W_{oh}^T(o_t - y_t) + W_{hh}^T\delta_{t+1}$$

在最终时间步 T：
$$\delta_T = W_{oh}^T(o_T - y_T)$$

展开 $\delta_t$ 到所有时间步：
$$\delta_t = \sum_{k=t}^{T} (W_{hh}^T)^{k-t} W_{oh}^T (o_k - y_k)$$

损失对 $W_{hh}$ 的梯度：
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t}\frac{\partial h_t}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t h_{t-1}^T$$

展开后：
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \sum_{k=t}^{T} (W_{hh}^T)^{k-t} W_{oh}^T (o_k - y_k) h_{t-1}^T$$

**梯度消失或爆炸的条件：**

在展开式中，存在因子 $(W_{hh}^T)^{k-t}$。当 $W_{hh}$ 的特征值的绝对值：

- **梯度爆炸**：如果 $|\lambda_i| > 1$，则 $(W_{hh}^T)^{k-t}$ 随 $k-t$ 增大呈指数增长，导致梯度爆炸。

- **梯度消失**：如果 $|\lambda_i| < 1$，则 $(W_{hh}^T)^{k-t}$ 随 $k-t$ 增大呈指数衰减，导致梯度消失。

当 $|\lambda_i| \approx 1$ 时，梯度可以稳定传播。

In [3]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hh, W_hx, b_h):
    """
    RNN单元前向传播
    
    参数:
        x_t: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hh: 隐藏状态权重，形状 (hidden_size, hidden_size)
        W_hx: 输入权重，形状 (hidden_size, input_size)
        b_h: 偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
    """
    # 计算隐藏状态：h_t = tanh(W_hh @ h_prev + W_hx @ x_t + b_h)
    h_t = np.tanh(np.dot(h_prev, W_hh.T) + np.dot(x_t, W_hx.T) + b_h)
    return h_t


def rnn_backward(x_t, h_prev, h_t, dh_next, W_hh, W_hx, b_h):
    """
    RNN单元单步反向传播
    
    参数:
        x_t: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        dh_next: 上游梯度（损失对h_t的梯度），形状 (batch_size, hidden_size)
        W_hh: 隐藏状态权重，形状 (hidden_size, hidden_size)
        W_hx: 输入权重，形状 (hidden_size, input_size)
        b_h: 偏置，形状 (hidden_size,)
    
    返回:
        dx_t: 损失对x_t的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对h_prev的梯度，形状 (batch_size, hidden_size)
        dW_hh: 损失对W_hh的梯度，形状 (hidden_size, hidden_size)
        dW_hx: 损失对W_hx的梯度，形状 (hidden_size, input_size)
        db_h: 损失对b_h的梯度，形状 (hidden_size,)
    """
    batch_size, hidden_size = h_t.shape
    
    # 计算tanh的导数：dtanh = 1 - tanh^2
    dtanh = 1 - h_t ** 2  # 形状 (batch_size, hidden_size)
    
    # dh_next 经过tanh的链式法则
    dh = dh_next * dtanh  # 形状 (batch_size, hidden_size)
    
    # 计算对各个参数的梯度
    # dh = d(h_t)/d(param)，其中 h_t = tanh(W_hh @ h_prev + W_hx @ x_t + b_h)
    # 令 z = W_hh @ h_prev + W_hx @ x_t + b_h, h_t = tanh(z)
    # dh = dL/dh_t * (1 - tanh^2(z)) = dL/dz
    # 因此 dL/dz = dh
    
    # dW_hh = dL/dz * dz/dW_hh = dh^T @ h_prev
    dW_hh = np.dot(dh.T, h_prev)  # 形状 (hidden_size, hidden_size)
    
    # dW_hx = dL/dz * dz/dW_hx = dh^T @ x_t
    dW_hx = np.dot(dh.T, x_t)  # 形状 (hidden_size, input_size)
    
    # db_h = sum(dh, axis=0)
    db_h = np.sum(dh, axis=0)  # 形状 (hidden_size,)
    
    # dh_prev = dL/dz * dz/dh_prev = dh @ W_hh
    dh_prev = np.dot(dh, W_hh)  # 形状 (batch_size, hidden_size)
    
    # dx_t = dL/dz * dz/dx_t = dh @ W_hx
    dx_t = np.dot(dh, W_hx)  # 形状 (batch_size, input_size)
    
    return dx_t, dh_prev, dW_hh, dW_hx, db_h


# 测试代码
if __name__ == "__main__":
    print("=" * 50)
    print("3.2 RNN编程题 - 测试")
    print("=" * 50)
    
    # 设置参数
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    # 随机初始化
    np.random.seed(42)
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
    W_hx = np.random.randn(hidden_size, input_size) * 0.1
    b_h = np.zeros(hidden_size)
    
    print(f"输入 x_t 形状: {x_t.shape}")
    print(f"上一隐藏状态 h_prev 形状: {h_prev.shape}")
    print(f"W_hh 形状: {W_hh.shape}")
    print(f"W_hx 形状: {W_hx.shape}")
    print(f"b_h 形状: {b_h.shape}")
    
    # 前向传播
    h_t = rnn_forward(x_t, h_prev, W_hh, W_hx, b_h)
    print(f"\n当前隐藏状态 h_t 形状: {h_t.shape}")
    print(f"h_t:\n{h_t}")
    
    # 反向传播（模拟上游梯度）
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hh, dW_hx, db_h = rnn_backward(
        x_t, h_prev, h_t, dh_next, W_hh, W_hx, b_h
    )
    
    print(f"\n反向传播结果:")
    print(f"dx_t 形状: {dx_t.shape}")
    print(f"dh_prev 形状: {dh_prev.shape}")
    print(f"dW_hh 形状: {dW_hh.shape}")
    print(f"dW_hx 形状: {dW_hx.shape}")
    print(f"db_h 形状: {db_h.shape}")
    
    # 数值梯度验证（使用有限差分）
    print("\n" + "-" * 30)
    print("数值梯度验证（近似检查）:")
    
    eps = 1e-5
    # 验证 dW_hh
    W_hh_flat = W_hh.flatten()
    grad_approx = np.zeros_like(W_hh_flat)
    
    for i in range(min(5, len(W_hh_flat))):  # 只检查前5个
        W_hh_copy = W_hh.copy()
        W_hh_copy.flat[i] += eps
        h_t_plus = rnn_forward(x_t, h_prev, W_hh_copy, W_hx, b_h)
        
        W_hh_copy = W_hh.copy()
        W_hh_copy.flat[i] -= eps
        h_t_minus = rnn_forward(x_t, h_prev, W_hh_copy, W_hx, b_h)
        
        # 使用简单的标量损失 L = sum(h_t^2) / 2
        L_plus = 0.5 * np.sum(h_t_plus ** 2)
        L_minus = 0.5 * np.sum(h_t_minus ** 2)
        grad_approx[i] = (L_plus - L_minus) / (2 * eps)
    
    # 计算解析梯度对应的损失梯度
    dh_next_approx = h_t  # dL/dh_t = h_t
    _, _, dW_hh_analytic, _, _ = rnn_backward(
        x_t, h_prev, h_t, dh_next_approx, W_hh, W_hx, b_h
    )
    dW_hh_flat = dW_hh_analytic.flatten()
    
    print(f"前5个参数的数值梯度: {grad_approx[:5]}")
    print(f"前5个参数的解析梯度: {dW_hh_flat[:5]}")
    print("梯度验证通过！" if np.allclose(grad_approx[:5], dW_hh_flat[:5], rtol=1e-4) else "梯度验证失败！")

3.2 RNN编程题 - 测试
输入 x_t 形状: (2, 3)
上一隐藏状态 h_prev 形状: (2, 4)
W_hh 形状: (4, 4)
W_hx 形状: (4, 3)
b_h 形状: (4,)

当前隐藏状态 h_t 形状: (2, 4)
h_t:
[[-0.29800225 -0.44289224 -0.11514275 -0.1291687 ]
 [-0.11272415  0.03473292  0.13676303  0.08558655]]

反向传播结果:
dx_t 形状: (2, 3)
dh_prev 形状: (2, 4)
dW_hh 形状: (4, 4)
dW_hx 形状: (4, 3)
db_h 形状: (4,)

------------------------------
数值梯度验证（近似检查）:
前5个参数的数值梯度: [-0.37724179 -0.15655583  0.10055175  0.0656067  -0.57830361]
前5个参数的解析梯度: [-0.37724179 -0.15655583  0.10055175  0.0656067  -0.57830361]
梯度验证通过！


### 4.1 理论计算题

假设一个深度双向RNN，有L层，每层隐藏单元数为H，输入维度为D，输出维度为O（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

**解答：**

对于深度双向RNN，每一层包含两个方向的RNN：前向（forward）和后向（backward）。

**每层每方向的参数：**

对于一个单向RNN层，输入维度为 $d_{in}$，隐藏单元数为 H：
- 输入到隐藏的权重：$H \times d_{in}$
- 隐藏到隐藏的权重：$H \times H$
- 偏置：$H$

**各层参数计算：**

**第1层（底层）：**
- 输入维度为 D
- 前向：$H \times D + H \times H + H = H(D + H + 1)$
- 后向：$H \times D + H \times H + H = H(D + H + 1)$
- 小计：$2H(D + H + 1)$

**第 $l$ 层（$2 \le l \le L$）：**
- 输入维度为 $2H$（来自上一层的双向拼接）
- 前向：$H \times (2H) + H \times H + H = H(2H + H + 1) = H(3H + 1)$
- 后向：$H \times (2H) + H \times H + H = H(3H + 1)$
- 小计：$2H(3H + 1)$

**输出层（最后输出层）：**
- 输入维度为 $2H$（最后一层双向拼接）
- 输出维度为 O
- 权重：$O \times 2H$
- 偏置：$O$
- 小计：$O(2H + 1)$

**总参数：**
$$\text{Total} = 2H(D + H + 1) + (L - 1) \times 2H(3H + 1) + O(2H + 1)$$

化简：
$$\text{Total} = 2H(D + H + 1) + 2H(L - 1)(3H + 1) + O(2H + 1)$$

$$= 2H\left[D + H + 1 + (L - 1)(3H + 1)\right] + O(2H + 1)$$

$$= 2H\left[D + H + 1 + (L - 1)(3H + 1)\right] + 2OH + O$$

In [4]:
import torch
import torch.nn as nn
import numpy as np

class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        """
        参数:
            input_dim: 输入维度
            hidden_dim: 隐藏单元数
            num_layers: RNN层数
            dropout: dropout比率
        """
        super(BidirectionalRNNEncoder, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 使用PyTorch的RNN实现双向
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False,  # 使用 (seq_len, batch, input_dim) 格式
            dropout=dropout if num_layers > 1 else 0.0
        )
    
    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        # RNN前向传播
        # outputs: (seq_len, batch, num_directions * hidden_dim)
        # h_n: (num_layers * num_directions, batch, hidden_dim)
        outputs, h_n = self.rnn(X)
        
        # outputs 已经是拼接后的状态 (seq_len, batch, 2*hidden_dim)
        # 取最后一个时间步的拼接状态作为序列表示
        # final_state = outputs[-1, :, :]  # (batch, 2*hidden_dim)
        final_state = outputs[-1]  # (batch, 2*hidden_dim)
        
        return outputs, final_state


class BidirectionalRNNEncoderManual(nn.Module):
    """
    手动实现的双向RNN编码器（用于教学演示）
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BidirectionalRNNEncoderManual, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 为每一层创建前向和后向RNN单元
        self.forward_rnns = nn.ModuleList()
        self.backward_rnns = nn.ModuleList()
        
        for layer in range(num_layers):
            if layer == 0:
                fwd_input_dim = input_dim
                bwd_input_dim = input_dim
            else:
                fwd_input_dim = 2 * hidden_dim  # 双向拼接
                bwd_input_dim = 2 * hidden_dim
            
            self.forward_rnns.append(
                nn.RNNCell(fwd_input_dim, hidden_dim)
            )
            self.backward_rnns.append(
                nn.RNNCell(bwd_input_dim, hidden_dim)
            )
    
    def forward(self, X):
        """
        手动实现双向RNN前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        seq_len, batch_size, _ = X.shape
        
        # 存储每一层的输出
        layer_outputs = []
        current_input = X
        
        for layer in range(self.num_layers):
            # 前向传播
            h_fwd = torch.zeros(batch_size, self.hidden_dim, device=X.device)
            fwd_outputs = []
            for t in range(seq_len):
                h_fwd = self.forward_rnns[layer](current_input[t], h_fwd)
                fwd_outputs.append(h_fwd)
            
            # 后向传播
            h_bwd = torch.zeros(batch_size, self.hidden_dim, device=X.device)
            bwd_outputs = []
            for t in range(seq_len - 1, -1, -1):
                h_bwd = self.backward_rnns[layer](current_input[t], h_bwd)
                bwd_outputs.insert(0, h_bwd)  # 反转回正序
            
            # 拼接前向和后向输出
            # 每个时间步: (batch, 2*hidden_dim)
            layer_out = torch.stack([
                torch.cat([fwd_outputs[t], bwd_outputs[t]], dim=1)
                for t in range(seq_len)
            ], dim=0)  # (seq_len, batch, 2*hidden_dim)
            
            layer_outputs.append(layer_out)
            current_input = layer_out  # 下一层的输入
        
        # 最后一层的输出
        outputs = layer_outputs[-1]  # (seq_len, batch, 2*hidden_dim)
        final_state = outputs[-1]  # (batch, 2*hidden_dim)
        
        return outputs, final_state


# 测试代码
if __name__ == "__main__":
    print("=" * 50)
    print("4.2 双向RNN编码器 - 测试")
    print("=" * 50)
    
    # 设置参数
    seq_len = 5
    batch_size = 3
    input_dim = 8
    hidden_dim = 4
    num_layers = 2
    
    # 创建随机输入
    X = torch.randn(seq_len, batch_size, input_dim)
    print(f"输入 X 形状: {X.shape}")
    
    # 使用PyTorch实现
    print("\n" + "-" * 30)
    print("使用 torch.nn.RNN 实现:")
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers)
    outputs, final_state = encoder(X)
    print(f"输出形状: {outputs.shape}")  # (seq_len, batch, 2*hidden_dim)
    print(f"最终状态形状: {final_state.shape}")  # (batch, 2*hidden_dim)
    print(f"最终状态:\n{final_state}")
    
    # 使用手动实现
    print("\n" + "-" * 30)
    print("手动实现:")
    encoder_manual = BidirectionalRNNEncoderManual(input_dim, hidden_dim, num_layers)
    # 复制权重（简单起见，不做权重复制）
    outputs_manual, final_state_manual = encoder_manual(X)
    print(f"输出形状: {outputs_manual.shape}")
    print(f"最终状态形状: {final_state_manual.shape}")
    print(f"最终状态:\n{final_state_manual}")
    
    # 检查每个时间步的拼接状态
    print("\n" + "-" * 30)
    print("每个时间步的输出:")
    for t in range(seq_len):
        print(f"时间步 {t}: 形状 {outputs[t].shape}")

4.2 双向RNN编码器 - 测试
输入 X 形状: torch.Size([5, 3, 8])

------------------------------
使用 torch.nn.RNN 实现:
输出形状: torch.Size([5, 3, 8])
最终状态形状: torch.Size([3, 8])
最终状态:
tensor([[-0.8071, -0.4237,  0.5218,  0.3519, -0.7269,  0.7407, -0.1623, -0.0720],
        [-0.7636,  0.8705,  0.4473, -0.3156, -0.1216,  0.0537, -0.6830,  0.6730],
        [-0.7996, -0.5407, -0.1198,  0.6061, -0.2484,  0.2982, -0.2865, -0.1222]],
       grad_fn=<SelectBackward0>)

------------------------------
手动实现:
输出形状: torch.Size([5, 3, 8])
最终状态形状: torch.Size([3, 8])
最终状态:
tensor([[-0.3894, -0.6896,  0.3248,  0.6757,  0.1347, -0.2133, -0.2848, -0.0408],
        [-0.3625, -0.5172,  0.7230, -0.3516,  0.4406, -0.4075, -0.5024,  0.0593],
        [ 0.6648, -0.8573,  0.2268,  0.3147, -0.3295, -0.1020,  0.7000,  0.0282]],
       grad_fn=<SelectBackward0>)

------------------------------
每个时间步的输出:
时间步 0: 形状 torch.Size([3, 8])
时间步 1: 形状 torch.Size([3, 8])
时间步 2: 形状 torch.Size([3, 8])
时间步 3: 形状 torch.Size([3, 8])
时间步 4: 形状 torch.Siz

### 5.1 理论计算题

在Skip-gram模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样 $K$ 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 $\mathbf{v}_c, \mathbf{u}_o$，负样本词向量为 $\mathbf{u}_{n_k}$，写出完整的目标函数。

**解答：**

在Skip-gram模型中，给定中心词 $w_c$，我们希望最大化上下文词 $w_o$ 出现的概率。

使用负采样，目标是将正样本（真实上下文）与负样本（随机采样的词）区分开。

**损失函数（对数似然）：**

对于每个正样本 $(w_c, w_o)$，负采样损失函数为：

$$L = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^{K} \mathbb{E}_{n_k \sim P_n(w)} \left[ \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c) \right]$$

其中 $\sigma(x) = \frac{1}{1 + e^{-x}}$ 是sigmoid函数。

完整的目标函数（对于单个正样本和 $K$ 个负样本）：

$$L = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)$$

$$= -\log \frac{1}{1 + e^{-\mathbf{u}_o^T \mathbf{v}_c}} - \sum_{k=1}^{K} \log \frac{1}{1 + e^{\mathbf{u}_{n_k}^T \mathbf{v}_c}}$$

$$= -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)$$

**负样本采样方法：**

负样本从噪声分布 $P_n(w)$ 中采样。常用的噪声分布是词频的3/4次方：

$$P_n(w) = \frac{\text{count}(w)^{3/4}}{\sum_{i} \text{count}(w_i)^{3/4}}$$

这种方法（称为"unigram distribution raised to the 3/4 power"）可以：
1. 降低高频词的采样概率
2. 提高低频词的采样概率
3. 使采样更加平滑

具体采样步骤：
1. 计算每个词的词频 count(w)
2. 计算 count(w)^{3/4}
3. 归一化得到概率分布 $P_n(w)$
4. 从该分布中独立采样 $K$ 个负样本（不包括正样本 $w_o$）

In [5]:
import numpy as np

def cbow_forward(context_indices, W, W_out):
    """
    CBOW模型前向传播和损失计算（完整softmax）
    
    参数:
        context_indices: 一批上下文词的索引列表，形状 (batch_size, context_size)
                        每个样本有context_size个上下文词
        W: 输入权重矩阵，形状 (V, d)，V=词汇表大小，d=嵌入维度
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
        loss: 交叉熵损失值（标量）
        probs: 输出概率分布，形状 (batch_size, V)
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 1. 获取每个上下文词的嵌入向量
    # context_indices: (batch_size, context_size)
    # 使用W[context_indices]获取嵌入: (batch_size, context_size, d)
    context_embeddings = W[context_indices]  # (batch_size, context_size, d)
    
    # 2. 计算平均上下文向量作为隐藏层
    # 对context_size维度求平均: (batch_size, d)
    h = np.mean(context_embeddings, axis=1)  # (batch_size, d)
    
    # 3. 计算输出得分
    # scores = h @ W_out: (batch_size, V)
    scores = np.dot(h, W_out)  # (batch_size, V)
    
    # 4. 计算softmax得到概率分布
    # 使用数值稳定性技巧: 减去最大值
    scores_max = np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)  # (batch_size, V)
    
    return probs, h, scores


def cbow_loss(context_indices, target_indices, W, W_out):
    """
    CBOW模型完整前向传播和损失计算
    
    参数:
        context_indices: 一批上下文词的索引列表，形状 (batch_size, context_size)
        target_indices: 目标中心词索引，形状 (batch_size,)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
        loss: 交叉熵损失值（标量）
        probs: 输出概率分布，形状 (batch_size, V)
    """
    batch_size = context_indices.shape[0]
    
    # 前向传播
    probs, h, scores = cbow_forward(context_indices, W, W_out)
    
    # 计算交叉熵损失
    # loss = -1/batch_size * sum(log(probs[i, target_indices[i]]))
    log_probs = np.log(probs + 1e-10)  # 加小值防止log(0)
    loss = -np.mean(log_probs[np.arange(batch_size), target_indices])
    
    return loss, probs


# 测试代码
if __name__ == "__main__":
    print("=" * 50)
    print("5.2 CBOW模型 - 测试")
    print("=" * 50)
    
    # 设置参数
    V = 10  # 词汇表大小
    d = 4   # 嵌入维度
    batch_size = 3
    context_size = 2
    
    # 随机初始化权重
    np.random.seed(42)
    W = np.random.randn(V, d) * 0.01
    W_out = np.random.randn(d, V) * 0.01
    
    # 生成模拟数据
    # 上下文词索引: (batch_size, context_size)
    context_indices = np.random.randint(0, V, size=(batch_size, context_size))
    # 目标词索引: (batch_size,)
    target_indices = np.random.randint(0, V, size=batch_size)
    
    print(f"词汇表大小 V = {V}")
    print(f"嵌入维度 d = {d}")
    print(f"批次大小 batch_size = {batch_size}")
    print(f"上下文大小 context_size = {context_size}")
    print(f"\nW 形状: {W.shape}")
    print(f"W_out 形状: {W_out.shape}")
    print(f"\n上下文索引:\n{context_indices}")
    print(f"目标索引: {target_indices}")
    
    # 计算损失
    loss, probs = cbow_loss(context_indices, target_indices, W, W_out)
    
    print(f"\n输出概率分布形状: {probs.shape}")
    print(f"概率分布 (每个样本的softmax输出):\n{probs}")
    print(f"\n交叉熵损失: {loss:.6f}")
    
    # 手动验证一个样本
    print("\n" + "-" * 30)
    print("手动验证第一个样本:")
    sample_idx = 0
    print(f"上下文索引: {context_indices[sample_idx]}")
    print(f"目标索引: {target_indices[sample_idx]}")
    
    # 获取上下文嵌入
    context_emb = W[context_indices[sample_idx]]  # (context_size, d)
    print(f"上下文嵌入:\n{context_emb}")
    h_sample = np.mean(context_emb, axis=0)  # (d,)
    print(f"平均上下文向量 (隐藏层):\n{h_sample}")
    
    scores_sample = np.dot(h_sample, W_out)  # (V,)
    print(f"得分 (前5个): {scores_sample[:5]}")
    
    # softmax
    exp_scores_sample = np.exp(scores_sample - np.max(scores_sample))
    probs_sample = exp_scores_sample / np.sum(exp_scores_sample)
    print(f"概率 (前5个): {probs_sample[:5]}")
    print(f"目标词概率: {probs_sample[target_indices[sample_idx]]:.6f}")
    print(f"负对数似然: {-np.log(probs_sample[target_indices[sample_idx]] + 1e-10):.6f}")
    print(f"平均损失: {loss:.6f}")

5.2 CBOW模型 - 测试
词汇表大小 V = 10
嵌入维度 d = 4
批次大小 batch_size = 3
上下文大小 context_size = 2

W 形状: (10, 4)
W_out 形状: (4, 10)

上下文索引:
[[4 6]
 [6 3]
 [6 2]]
目标索引: [5 1 9]

输出概率分布形状: (3, 10)
概率分布 (每个样本的softmax输出):
[[0.0999952  0.09998899 0.10000824 0.10000505 0.10001612 0.09998657
  0.0999993  0.09997954 0.09999103 0.10002995]
 [0.10000367 0.1000056  0.10002337 0.10001184 0.09998486 0.09997358
  0.10001037 0.09998819 0.09999236 0.10000616]
 [0.09999894 0.0999966  0.10000524 0.1000104  0.10000341 0.0999933
  0.09999806 0.09998359 0.0999943  0.10001615]]

交叉熵损失: 2.302557

------------------------------
手动验证第一个样本:
上下文索引: [4 6]
目标索引: 5
上下文嵌入:
[[-0.01012831  0.00314247 -0.00908024 -0.01412304]
 [-0.00544383  0.00110923 -0.01150994  0.00375698]]
平均上下文向量 (隐藏层):
[-0.00778607  0.00212585 -0.01029509 -0.00518303]
得分 (前5个): [-2.00078075e-05 -8.21322399e-05  1.10369147e-04  7.85020627e-05
  1.89168128e-04]
概率 (前5个): [0.0999952  0.09998899 0.10000824 0.10000505 0.10001612]
目标词概率: 0.099987
负对数似然: 2.302719
平均损失:

### 6.1 理论计算题

给定查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$，键矩阵 $K \in \mathbb{R}^{3 \times 4}$，值矩阵 $V \in \mathbb{R}^{3 \times 5}$。计算缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤（先计算得分矩阵，再 softmax，再加权求和）。使用 score = $Q K^T / \sqrt{d_k}$（$d_k = 4$）。可以只列出数值计算过程（用符号或具体数值）。

**解答：**

给定：
- $Q \in \mathbb{R}^{2 \times 4}$（2个查询，每个维度为4）
- $K \in \mathbb{R}^{3 \times 4}$（3个键，每个维度为4）
- $V \in \mathbb{R}^{3 \times 5}$（3个值，每个维度为5）
- $d_k = 4$

**步骤1：计算得分矩阵（注意力分数）**

$$S = \frac{Q K^T}{\sqrt{d_k}}$$

其中 $Q K^T \in \mathbb{R}^{2 \times 3}$：

$$Q K^T = \begin{bmatrix}
q_1 \cdot k_1 & q_1 \cdot k_2 & q_1 \cdot k_3 \\
q_2 \cdot k_1 & q_2 \cdot k_2 & q_2 \cdot k_3
\end{bmatrix}$$

$$\sqrt{d_k} = \sqrt{4} = 2$$

因此：
$$S = \frac{1}{2} \begin{bmatrix}
q_1 \cdot k_1 & q_1 \cdot k_2 & q_1 \cdot k_3 \\
q_2 \cdot k_1 & q_2 \cdot k_2 & q_2 \cdot k_3
\end{bmatrix} = \begin{bmatrix}
s_{11} & s_{12} & s_{13} \\
s_{21} & s_{22} & s_{23}
\end{bmatrix}$$

**步骤2：对每一行应用softmax（沿键维度）**

对于第 $i$ 行（第 $i$ 个查询）：

$$\alpha_{ij} = \text{softmax}(s_{ij}) = \frac{e^{s_{ij}}}{\sum_{j'=1}^{3} e^{s_{ij'}}}$$

得到注意力权重矩阵：
$$A = \begin{bmatrix}
\alpha_{11} & \alpha_{12} & \alpha_{13} \\
\alpha_{21} & \alpha_{22} & \alpha_{23}
\end{bmatrix} \in \mathbb{R}^{2 \times 3}$$

其中每行之和为1：$\sum_{j} \alpha_{ij} = 1$

**步骤3：加权求和得到输出**

$$\text{Output} = A \times V$$

$$\text{Output} = \begin{bmatrix}
\alpha_{11} & \alpha_{12} & \alpha_{13} \\
\alpha_{21} & \alpha_{22} & \alpha_{23}
\end{bmatrix} \begin{bmatrix}
v_1^T \\
v_2^T \\
v_3^T
\end{bmatrix}$$

其中 $v_j \in \mathbb{R}^{5}$ 是 $V$ 的第 $j$ 行（列向量形式）。

具体计算：

对于第 $i$ 个查询的输出（第 $i$ 行）：
$$\text{Output}_i = \sum_{j=1}^{3} \alpha_{ij} \cdot v_j$$

因此最终输出：
$$\text{Output} = \begin{bmatrix}
\alpha_{11}v_1 + \alpha_{12}v_2 + \alpha_{13}v_3 \\
\alpha_{21}v_1 + \alpha_{22}v_2 + \alpha_{23}v_3
\end{bmatrix} \in \mathbb{R}^{2 \times 5}$$

**数值示例：**

假设具体数值：
$$Q = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1
\end{bmatrix}, \quad
K = \begin{bmatrix}
1 & 1 & 0 & 0 \\
0 & 1 & 1 & 0 \\
0 & 0 & 1 & 1
\end{bmatrix}, \quad
V = \begin{bmatrix}
1 & 2 & 3 & 4 & 5 \\
6 & 7 & 8 & 9 & 10 \\
11 & 12 & 13 & 14 & 15
\end{bmatrix}$$

步骤1：
$$Q K^T = \begin{bmatrix}
1 & 1 & 0 \\
1 & 1 & 1
\end{bmatrix}, \quad S = \frac{1}{2} \begin{bmatrix}
1 & 1 & 0 \\
1 & 1 & 1
\end{bmatrix} = \begin{bmatrix}
0.5 & 0.5 & 0 \\
0.5 & 0.5 & 0.5
\end{bmatrix}$$

步骤2（softmax）：
$$A = \begin{bmatrix}
\frac{e^{0.5}}{e^{0.5}+e^{0.5}+e^0} & \frac{e^{0.5}}{e^{0.5}+e^{0.5}+e^0} & \frac{e^0}{e^{0.5}+e^{0.5}+e^0} \\
\frac{e^{0.5}}{e^{0.5}+e^{0.5}+e^{0.5}} & \frac{e^{0.5}}{e^{0.5}+e^{0.5}+e^{0.5}} & \frac{e^{0.5}}{e^{0.5}+e^{0.5}+e^{0.5}}
\end{bmatrix}$$

$$A = \begin{bmatrix}
0.386 & 0.386 & 0.228 \\
0.333 & 0.333 & 0.333
\end{bmatrix}$$

步骤3（加权求和）：
$$\text{Output} = A \times V = \begin{bmatrix}
0.386 & 0.386 & 0.228 \\
0.333 & 0.333 & 0.333
\end{bmatrix} \begin{bmatrix}
1 & 2 & 3 & 4 & 5 \\
6 & 7 & 8 & 9 & 10 \\
11 & 12 & 13 & 14 & 15
\end{bmatrix}$$

$$\text{Output} = \begin{bmatrix}
0.386 \cdot 1 + 0.386 \cdot 6 + 0.228 \cdot 11 & \cdots & 0.386 \cdot 5 + 0.386 \cdot 10 + 0.228 \cdot 15 \\
0.333 \cdot 1 + 0.333 \cdot 6 + 0.333 \cdot 11 & \cdots & 0.333 \cdot 5 + 0.333 \cdot 10 + 0.333 \cdot 15
\end{bmatrix}$$

$$\text{Output} = \begin{bmatrix}
5.164 & 6.164 & 7.164 & 8.164 & 9.164 \\
6.000 & 7.000 & 8.000 & 9.000 & 10.000
\end{bmatrix}$$

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制 (Multi-Head Attention)
    
    参数:
        d_model: 模型维度
        num_heads: 注意力头数
        dropout: dropout比率（可选）
    """
    def __init__(self, d_model, num_heads, dropout=0.0):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # 最终输出线性层
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, X, mask=None):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, d_model)
            mask: 注意力掩码，形状 (seq_len, seq_len) 或 (batch, seq_len, seq_len)，可选
        
        返回:
            output: 输出，形状 (seq_len, batch, d_model)
            attention_weights: 注意力权重，形状 (batch, num_heads, seq_len, seq_len)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影得到 Q, K, V
        # 形状: (seq_len, batch, d_model)
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 2. 重塑为多头形式
        # (seq_len, batch, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_v).permute(1, 2, 0, 3)
        
        # 3. 缩放点积注意力
        # 计算得分: Q @ K^T / sqrt(d_k)
        # Q: (batch, num_heads, seq_len, d_k)
        # K: (batch, num_heads, seq_len, d_k)
        # scores: (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        # 应用掩码（如果提供）
        if mask is not None:
            # 将掩码中为0的位置设为负无穷
            if mask.dim() == 2:
                # mask形状: (seq_len, seq_len)，扩展到 (batch, num_heads, seq_len, seq_len)
                mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax得到注意力权重
        attention_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq_len, seq_len)
        attention_weights = self.dropout(attention_weights)
        
        # 加权求和: attention_weights @ V
        # V: (batch, num_heads, seq_len, d_v)
        # output: (batch, num_heads, seq_len, d_v)
        context = torch.matmul(attention_weights, V)
        
        # 4. 拼接所有头的输出
        # (batch, num_heads, seq_len, d_v) -> (batch, seq_len, num_heads * d_v) -> (seq_len, batch, d_model)
        context = context.permute(0, 2, 1, 3).contiguous()  # (batch, seq_len, num_heads, d_v)
        context = context.view(batch_size, seq_len, self.d_model)  # (batch, seq_len, d_model)
        context = context.permute(1, 0, 2)  # (seq_len, batch, d_model)
        
        # 5. 最终线性层
        output = self.W_o(context)  # (seq_len, batch, d_model)
        
        return output, attention_weights


class MultiHeadAttentionManual(nn.Module):
    """
    手动实现的多头注意力（不使用nn.Linear，用于教学演示）
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttentionManual, self).__init__()
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 初始化权重参数
        # Q, K, V 投影权重: (d_model, d_model)
        self.W_q = nn.Parameter(torch.randn(d_model, d_model) * 0.01)
        self.W_k = nn.Parameter(torch.randn(d_model, d_model) * 0.01)
        self.W_v = nn.Parameter(torch.randn(d_model, d_model) * 0.01)
        self.W_o = nn.Parameter(torch.randn(d_model, d_model) * 0.01)
        
    def forward(self, X):
        """
        手动实现多头注意力前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, d_model)
        
        返回:
            output: 输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影得到 Q, K, V
        # X: (seq_len, batch, d_model)
        # W_q: (d_model, d_model)
        # 使用矩阵乘法: (seq_len, batch, d_model) @ (d_model, d_model) -> (seq_len, batch, d_model)
        Q = torch.matmul(X, self.W_q)
        K = torch.matmul(X, self.W_k)
        V = torch.matmul(X, self.W_v)
        
        # 2. 重塑为多头形式
        # (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_v)
        
        # 交换维度: (batch, num_heads, seq_len, d_k)
        Q = Q.permute(1, 2, 0, 3)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)
        
        # 3. 缩放点积注意力
        # scores: (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        # Softmax
        attention_weights = F.softmax(scores, dim=-1)
        
        # 加权求和
        # context: (batch, num_heads, seq_len, d_v)
        context = torch.matmul(attention_weights, V)
        
        # 4. 拼接所有头的输出
        # (batch, seq_len, num_heads, d_v)
        context = context.permute(0, 2, 1, 3).contiguous()
        # (batch, seq_len, d_model)
        context = context.view(batch_size, seq_len, self.d_model)
        # (seq_len, batch, d_model)
        context = context.permute(1, 0, 2)
        
        # 5. 最终线性层
        output = torch.matmul(context, self.W_o)
        
        return output


def scaled_dot_product_attention_single_head(Q, K, V, d_k, mask=None):
    """
    单头缩放点积注意力（辅助函数）
    
    参数:
        Q: 查询，形状 (seq_len, d_k)
        K: 键，形状 (seq_len, d_k)
        V: 值，形状 (seq_len, d_v)
        d_k: 键维度
        mask: 掩码，可选
    
    返回:
        output: 输出，形状 (seq_len, d_v)
        attention_weights: 注意力权重，形状 (seq_len, seq_len)
    """
    # 计算得分
    scores = torch.matmul(Q, K.T) / (d_k ** 0.5)  # (seq_len, seq_len)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Softmax
    attention_weights = F.softmax(scores, dim=-1)  # (seq_len, seq_len)
    
    # 加权求和
    output = torch.matmul(attention_weights, V)  # (seq_len, d_v)
    
    return output, attention_weights


# ============ 测试代码 ============
if __name__ == "__main__":
    print("=" * 60)
    print("6.2 多头注意力 (Multi-Head Attention) - 测试")
    print("=" * 60)
    
    # 设置参数
    d_model = 4
    num_heads = 2
    seq_len = 3
    batch_size = 2
    
    print(f"d_model = {d_model}")
    print(f"num_heads = {num_heads}")
    print(f"每个头的维度 d_k = d_v = {d_model // num_heads}")
    print(f"序列长度 seq_len = {seq_len}")
    print(f"批次大小 batch_size = {batch_size}")
    
    # 创建随机输入
    torch.manual_seed(42)
    X = torch.randn(seq_len, batch_size, d_model)
    print(f"\n输入 X 形状: {X.shape}")
    print(f"X:\n{X}")
    
    # ========== 使用PyTorch实现 ==========
    print("\n" + "-" * 40)
    print("1. 使用 PyTorch nn.Module 实现:")
    
    mha = MultiHeadAttention(d_model, num_heads)
    output, attention_weights = mha(X)
    
    print(f"输出形状: {output.shape}")  # (seq_len, batch, d_model)
    print(f"输出:\n{output}")
    print(f"\n注意力权重形状: {attention_weights.shape}")  # (batch, num_heads, seq_len, seq_len)
    print(f"注意力权重 (第一个样本, 第一个头):\n{attention_weights[0, 0]}")
    
    # ========== 手动实现 ==========
    print("\n" + "-" * 40)
    print("2. 手动实现 (用于验证):")
    
    mha_manual = MultiHeadAttentionManual(d_model, num_heads)
    output_manual = mha_manual(X)
    print(f"输出形状: {output_manual.shape}")
    print(f"输出:\n{output_manual}")
    
    # ========== 单头注意力演示 ==========
    print("\n" + "-" * 40)
    print("3. 单头缩放点积注意力演示:")
    
    # 创建一个简单的例子
    d_k_single = 4
    d_v_single = 5
    seq_len_single = 3
    
    Q_single = torch.randn(seq_len_single, d_k_single)
    K_single = torch.randn(seq_len_single, d_k_single)
    V_single = torch.randn(seq_len_single, d_v_single)
    
    print(f"Q 形状: {Q_single.shape}")
    print(f"K 形状: {K_single.shape}")
    print(f"V 形状: {V_single.shape}")
    
    output_single, attn_weights_single = scaled_dot_product_attention_single_head(
        Q_single, K_single, V_single, d_k_single
    )
    
    print(f"\n注意力得分 (Q @ K^T / sqrt(d_k)):")
    scores_single = torch.matmul(Q_single, K_single.T) / (d_k_single ** 0.5)
    print(scores_single)
    
    print(f"\n注意力权重 (softmax):")
    print(attn_weights_single)
    print(f"每行之和: {attn_weights_single.sum(dim=1)}")
    
    print(f"\n输出形状: {output_single.shape}")
    print(f"输出:\n{output_single}")
    
    # ========== 验证多头注意力的拼接 ==========
    print("\n" + "-" * 40)
    print("4. 验证多头注意力的拼接过程:")
    
    # 使用一个简单的例子来展示拼接
    d_model_small = 4
    num_heads_small = 2
    d_k_small = d_model_small // num_heads_small  # 2
    seq_len_small = 3
    batch_small = 1
    
    X_small = torch.randn(seq_len_small, batch_small, d_model_small)
    
    mha_small = MultiHeadAttention(d_model_small, num_heads_small)
    output_small, attn_small = mha_small(X_small)
    
    print(f"输入形状: {X_small.shape}")
    print(f"输出形状: {output_small.shape}")
    print(f"注意力权重形状: {attn_small.shape}")
    
    # 展示每个头的维度
    print(f"\n每个头的维度: d_k = {d_k_small}, d_v = {d_k_small}")
    print(f"num_heads = {num_heads_small}")
    print(f"拼接后维度: {num_heads_small} * {d_k_small} = {d_model_small}")

6.2 多头注意力 (Multi-Head Attention) - 测试
d_model = 4
num_heads = 2
每个头的维度 d_k = d_v = 2
序列长度 seq_len = 3
批次大小 batch_size = 2

输入 X 形状: torch.Size([3, 2, 4])
X:
tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055],
         [ 0.6784, -1.2345, -0.0431, -1.6047]],

        [[ 0.3559, -0.6866, -0.4934,  0.2415],
         [-1.1109,  0.0915, -2.3169, -0.2168]],

        [[-0.3097, -0.3957,  0.8034, -0.6216],
         [-0.5920, -0.0631, -0.8286,  0.3309]]])

----------------------------------------
1. 使用 PyTorch nn.Module 实现:
输出形状: torch.Size([3, 2, 4])
输出:
tensor([[[-0.2397, -0.2013, -0.1152, -0.4099],
         [-0.2113,  0.0121,  0.2835, -0.2767]],

        [[-0.2822, -0.2615, -0.1421, -0.4736],
         [-0.1520, -0.0290,  0.1019, -0.2596]],

        [[-0.2301, -0.2152, -0.0427, -0.3770],
         [-0.1889,  0.0789,  0.3545, -0.2062]]], grad_fn=<UnsafeViewBackward0>)

注意力权重形状: torch.Size([2, 2, 3, 3])
注意力权重 (第一个样本, 第一个头):
tensor([[0.2043, 0.4566, 0.3392],
        [0.3582, 0.3059, 0.3359],
        [0